# 27 — Neural Autoencoder WJ 512

A model-based compressor baseline. It trains an encoder/decoder to reconstruct normalized quadtree vectors and adds a small Weighted-Jaccard calibration term on anchor-positive pairs.

This separates “neural compression” from the stronger triplet ranking objective.


In [ ]:
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

sys.path.append('/raid/ruban/hpmlproj/term_project')
from sota_experiment_common import (
    cleanup,
    eval_recall,
    l1_simplex,
    load_dataset,
    nmslib_neighbors,
    preload_rerank_corpus,
    release_rerank_corpus,
    rerank_wj_gpu,
    save_result,
)

# Edit here
dataset_name = "full"
out_dim = 512
device_str = "cuda:0"
device = torch.device(device_str if torch.cuda.is_available() else "cpu")
THREADS = 150
seed = 42
batch_size = 2048
epochs = 30
lr = 1e-3
weight_decay = 1e-4
candidate_ks = [500, 1000] if dataset_name == "10k" else [1000, 2000]
run_rerank = True

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print(f"device={device}")

METHOD_NAME = "autoencoder_wj_recon_512"
NOTEBOOK_NAME = "27_autoencoder_wj_recon_512.ipynb"
OUT_PATH = "/tmp/results_sota_autoencoder_wj_512.pkl"
CKPT_PATH = "/tmp/best_sota_autoencoder_wj_512_full.pt"


In [ ]:
qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset(dataset_name)
qt_norm = l1_simplex(qt.copy())


In [ ]:
def wj_torch(a, b):
    mins = torch.minimum(a, b).sum(dim=-1)
    maxs = torch.maximum(a, b).sum(dim=-1).clamp(min=1e-10)
    return mins / maxs

class PairDataset(Dataset):
    def __init__(self, qt, gt, query_start, max_pos=30):
        self.vecs = torch.tensor(qt, dtype=torch.float32)
        self.pairs = []
        for qid, neighbors in gt.items():
            for nid in neighbors[:max_pos]:
                if qid >= query_start and nid < query_start:
                    self.pairs.append((qid, nid))
        random.shuffle(self.pairs)
        print(f"pairs={len(self.pairs):,}")
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        qid, pid = self.pairs[idx]
        rid = random.randrange(0, query_start)
        return self.vecs[qid], self.vecs[pid], self.vecs[rid]

def embed_all(model, qt, batch_size=512):
    enc = model.module if hasattr(model, "module") else model
    enc.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(qt), batch_size):
            x = torch.tensor(qt[start:start + batch_size], dtype=torch.float32, device=device)
            out.append(enc.encode(x).detach().cpu().numpy().astype(np.float32))
    return np.vstack(out)

def eval_embeddings(embs, method_name, out_path, notebook_name):
    corpus_embs = embs[:query_start]
    query_embs = embs[query_start:]
    max_k = max(max(candidate_ks), 500)
    print(f"embs={embs.shape} | mem={corpus_embs.nbytes/1024**2:.1f} MB")
    nbrs, info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=max_k, threads=THREADS)
    metrics = {**eval_recall(gt, nbrs, query_start, max_k), **info, "dim": out_dim, "vec_mb": corpus_embs.nbytes/1024**2}
    for k, v in metrics.items():
        if isinstance(k, int): print(f"R@{k:<4} = {v:.4f}")
    print(f"QPS={metrics['qps']:.1f}")
    save_result(out_path, dataset_name, method_name, metrics, meta={"notebook": notebook_name})
    if run_rerank:
        preload_rerank_corpus(corpus_qt, corpus_sums)
        for ck in candidate_ks:
            cand, cand_info = nmslib_neighbors(corpus_embs, query_embs, space="WeightedJaccard", k=ck, threads=THREADS)
            t0 = time.time()
            rr = rerank_wj_gpu(query_qt, cand, corpus_qt, corpus_sums, top_k=ck, batch_size=8)
            qps_total = len(query_qt) / max(time.time() - t0 + len(query_qt)/max(cand_info['qps'], 1e-9), 1e-9)
            rr_metrics = {**eval_recall(gt, rr, query_start, ck), "qps": qps_total, "candidate_k": ck}
            key = f"{method_name}_rerank_{ck}"
            for k, v in rr_metrics.items():
                if isinstance(k, int): print(f"{key} R@{k:<4} = {v:.4f}")
            save_result(out_path, dataset_name, key, rr_metrics, meta={"notebook": notebook_name})
        release_rerank_corpus()


In [ ]:
class WJAutoencoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 4096), nn.ReLU(),
            nn.Linear(4096, in_dim), nn.ReLU(),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        z = self.encode(x)
        rec = self.decoder(z)
        rec = rec / rec.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return z, rec

# DataLoader returns only integer indices — no large tensor transfer per step.
class IndexReconDataset(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, idx): return idx

# Pre-load all vectors to GPU once (~17 GB); eliminates 150 MB PCIe transfer per step.
print("Pre-loading vectors to GPU...")
vecs_gpu = torch.from_numpy(np.ascontiguousarray(qt_norm, dtype=np.float32)).to(device)
print(f"Loaded: {vecs_gpu.nbytes/1024**3:.2f} GB on {device}")

model = WJAutoencoder(qt.shape[1], out_dim)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
    model = nn.DataParallel(model)
model = model.to(device)
loader = DataLoader(IndexReconDataset(len(qt_norm)), batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=True, persistent_workers=True)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
best = float('inf')
t_start = time.time()
for epoch in range(1, epochs + 1):
    model.train(); total = 0; steps = 0
    for idx in loader:
        x = vecs_gpu[idx.to(device)]
        z, rec = model(x)
        loss = F.mse_loss(rec, x)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        total += float(loss.detach()); steps += 1
    avg = total / max(steps, 1)
    if avg < best:
        best = avg; torch.save(model.module.state_dict() if hasattr(model, "module") else model.state_dict(), CKPT_PATH)
    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        elapsed = time.time() - t_start
        eta = elapsed / epoch * (epochs - epoch)
        print(f"epoch {epoch:02d}/{epochs} loss={avg:.3e}  best={best:.3e}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
print(f"best={best:.3e} saved {CKPT_PATH}")


In [ ]:
(model.module if hasattr(model, "module") else model).load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
embs = embed_all(model, qt_norm, batch_size=512)
eval_embeddings(embs, METHOD_NAME, OUT_PATH, NOTEBOOK_NAME)
cleanup()
